# **Test Ligand Class**


In [ ]:
from src.ligand import Ligand


smile = "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"
sdf_path = ".\\data\\raw\\P-L\\2001-2010\\1d4h\\1d4h_ligand.sdf"
ligand_smile = Ligand.from_smile(smile)  
ligand_sdf = Ligand.from_sdf(sdf_path)

ligand_sdf.visualize_py3Dmol()
ligand_smile.visualize_py3Dmol()


In [ ]:
x, edge_index, edge_attr, pos = ligand_smile.get_pyg_graph()


print(f"x: {x.shape}, edge_index: {edge_index.shape}, edge_attr: {edge_attr.shape}, pos: {pos.shape}")

edge_index


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

ligand_sdf_smile = Ligand.from_smile(ligand_sdf.to_smile())

edge_index_sdf = ligand_sdf._get_edge_index()
edge_index_smile = ligand_sdf_smile._get_edge_index()

print("Edge Index SDF:")
print(edge_index_sdf)   
print("\nEdge Index SMILE:")
print(edge_index_smile)

#costruisco due grafi nx dagli edge_index sono in formato COO, devo convertirli in formato adiacenza
def edge_index_to_nx(edge_index):
    G = nx.Graph()
    for i in range(edge_index.shape[1]):
        G.add_edge(edge_index[0, i].item(), edge_index[1, i].item())
    return G

G_sdf = edge_index_to_nx(edge_index_sdf)
G_smile = edge_index_to_nx(edge_index_smile)

#plotto i grafi e verifico che siano isomorfi

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
nx.draw(G_sdf, with_labels=True, node_color='lightblue', edge_color='gray')
plt.title("Grafo da SDF")
plt.subplot(1, 2, 2)
nx.draw(G_smile, with_labels=True, node_color='lightgreen', edge_color='gray')
plt.title("Grafo da SMILE")
plt.show()


In [ ]:
from networkx.algorithms import isomorphism

# Verifica semplice isomorfismo
are_isomorphic = nx.is_isomorphic(G_sdf, G_smile)

print(f"I due grafi sono isomorfi? {are_isomorphic}")

if are_isomorphic:
    NM = isomorphism.GraphMatcher(G_sdf, G_smile)
    if NM.is_isomorphic():
        print("Mappa di corrispondenza atomi (SDF -> SMILE):")
        print(NM.mapping)

## Verifica correttezza features

In [ ]:
# verifico che le feature degli atomi
print(f"x.shape: {x.shape} - numero di atomi: {x.shape[0]}, numero di feature per atomo: {x.shape[1]}")
print("x [numero_atomico, grado, H_totali, carica_formale, aromaticità, anello, ibridazione]:")
x_smile = ligand_smile._get_node_features()
print(x_smile)

mol=ligand_smile.mol
for atom in mol.GetAtoms():
    print(f"Atomo {atom.GetIdx()}: {atom.GetSymbol()}, grado: {atom.GetDegree()}, H_totali: {atom.GetTotalNumHs()}, carica formale: {atom.GetFormalCharge()}, aromaticità: {atom.GetIsAromatic()}, anello: {atom.IsInRing()}, ibridazione: {atom.GetHybridization()}")